In [1]:
# Here derived MMS additonal terms for case of homogenous solid with Oxyz reference axes coinciding with O123 material 
# axes. Linear elasticity with Duhamel-Neumann thermal term is considered. Damage kinetic equation is choosen to have
# Rabotnov form: d\omega / dT = A * ((\sigma[3][3] - \sigma_th)/sigma_th)^m * (1/(1-\omega))^m
#
# Unfortunatly, the current implementation of the C++ code does not support automatic MMS testing,
# so boundary and initial conditions, additional body force and kinetic eqation additional term should be changed manually 
# for each specific MMS test

In [1]:
reset()

In [2]:
from sympy.utilities.codegen import codegen
from sympy import ccode
from IPython.display import display, Markdown
import re

In [3]:

var('x, y, z, T')                # spatial coordinates and temperature
var('A, m, sigma_th')            # damage kinetic equation parameters
var('beta_0, beta_lin, C11_0, C33_0, C44_0, C12_0, C13_0')          #elastic material parameteres
var('alpha_11, alpha_22, alpha_33') #thermal expansion material parameters
var('T_0', 'T_end', 'omega_end') # solution parameteres, not a model parameters
var('L')                         # geometric parameteres

L

In [4]:
#t = (T_0 - T) / (T_0 - T_end) 
var('t') 

t

In [5]:
omega = 0.5*sqrt(t)  #defined damage as fuction of temperature and spatial variables
show("predefined damage: ", omega)

'predefined damage: ' 0.500000000000000*sqrt(t)

In [6]:
u_x = cos(pi*x/L) * cos(pi*y/L) * cos(pi*z/L) * exp(-t) #defined displacment components
u_y = sin(pi*x/L) * cos(pi*y/L) * cos(pi*z/L) * exp(-t) 
u_z = sin(pi*x/L) * sin(pi*y/L) * cos(pi*z/L) * exp(-t) 
u = vector([u_x, u_y, u_z])
show("predefined displacements: ", u)

'predefined displacements: ' (cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t), cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L), cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L))

In [7]:
grad_u = matrix(SR, 3, 3)  # symbolic ring
for i in range(3):
    for j in range(3):
        grad_u[i,j] = diff(u[i], [x, y, z][j])  # creating matrix of displacement gradient

In [8]:
fullEpsilon = (grad_u + grad_u.T) / 2

In [9]:
cteTensor = matrix (SR, 3, 3)
cteTensor[0,0] = alpha_11
cteTensor[1,1] = alpha_11
cteTensor[2,2] = alpha_33

show("CTE tensor:", cteTensor)

thermalEpsilon = cteTensor * (T-T_0)

'CTE tensor:' [alpha_11        0        0]
[       0 alpha_11        0]
[       0        0 alpha_33]

In [10]:
elasticEpsilon = fullEpsilon - thermalEpsilon
show("Strain ε:", elasticEpsilon)
strain_voigt = vector(SR,[elasticEpsilon[0][0], elasticEpsilon[1][1], elasticEpsilon[2][2], elasticEpsilon[1][2], elasticEpsilon[0][2], elasticEpsilon[0][1]])

'Strain ε:' [                                    -pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L - (T - T_0)*alpha_11 1/2*pi*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L - 1/2*pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L 1/2*pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L - 1/2*pi*cos(pi*x/L)*cos(pi*y/L)*e^(-t)*sin(pi*z/L)/L]
[1/2*pi*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L - 1/2*pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L                                     -pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L - (T - T_0)*alpha_11 1/2*pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L - 1/2*pi*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L]
[1/2*pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L - 1/2*pi*cos(pi*x/L)*cos(pi*y/L)*e^(-t)*sin(pi*z/L)/L 1/2*pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L - 1/2*pi*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L                                     -pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L - (T - T_0)*alpha_33]

In [11]:
beta = 1  #isotropic elasticity temperature dependency function 
C11 = C11_0 * beta
C33 = C33_0 * beta * (1 - omega)
C44 = C44_0* beta
C12 = C12_0 * beta
C13 = C13_0 * beta
C66 = (C11 - C12) / 2

In [12]:
C_matrix = matrix(SR, [
    [C11, C12, C13,   0,   0,    0],
    [C12, C11, C13,   0,   0,    0],
    [C13, C13, C33,   0,   0,    0],
    [  0,   0,   0, C44,   0,    0],
    [  0,   0,   0,   0, C44,    0],
    [  0,   0,   0,   0,   0,  C66]
])
show("Elastic matrix in Voigt notation:", C_matrix)

'Elastic matrix in Voigt notation:' [                                 C11_0                                  C12_0                                  C13_0                                      0                                      0                                      0]
[                                 C12_0                                  C11_0                                  C13_0                                      0                                      0                                      0]
[                                 C13_0                                  C13_0 C33_0*(-0.500000000000000*sqrt(t) + 1)                                      0                                      0                                      0]
[                                     0                                      0                                      0                                  C44_0                                      0                                      0]
[                                     0                                      0                                      0                                      0                                  C44_0                                      0]
[                                     0                                      0                                      0                                      0                                      0                  1/2*C11_0 - 1/2*C12_0]

In [13]:
stress_voigt = C_matrix * strain_voigt
show("Stress in Voigt notation:", stress_voigt)

'Stress in Voigt notation:' (-(pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L + (T - T_0)*alpha_11)*C11_0 - (pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L + (T - T_0)*alpha_11)*C12_0 - (pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L + (T - T_0)*alpha_33)*C13_0, -(pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L + (T - T_0)*alpha_11)*C11_0 - (pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L + (T - T_0)*alpha_11)*C12_0 - (pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L + (T - T_0)*alpha_33)*C13_0, -(pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L + (T - T_0)*alpha_33)*C33_0*(-0.500000000000000*sqrt(t) + 1) - (pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L + (T - T_0)*alpha_11)*C13_0 - (pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L + (T - T_0)*alpha_11)*C13_0, 1/2*(pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L - pi*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L)*C44_0, 1/2*(pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L - pi*cos(pi*x/L)*cos(pi*y/L)*e^(-t)*sin(pi*z/L)/L)*C44_0, 1/4*(pi*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L - pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L)*(C11_0 - C12_0))

In [15]:
stress = matrix(SR, 3, 3)  # symbolic ring
stress[0,0] = stress_voigt[0]
stress[1,1] = stress_voigt[1]
stress[2,2] = stress_voigt[2]

stress[1,2] = stress_voigt[3]
stress[2,1] = stress_voigt[3]

stress[0,2] = stress_voigt[4]
stress[2,0] = stress_voigt[4]

stress[0,1] = stress_voigt[5]
stress[1,0] = stress_voigt[5]

show("Stress", stress)


'Stress' [                                 -(pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L + (T - T_0)*alpha_11)*C11_0 - (pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L + (T - T_0)*alpha_11)*C12_0 - (pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L + (T - T_0)*alpha_33)*C13_0                                                                                                                                                      1/4*(pi*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L - pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L)*(C11_0 - C12_0)                                                                                                                                                                1/2*(pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L - pi*cos(pi*x/L)*cos(pi*y/L)*e^(-t)*sin(pi*z/L)/L)*C44_0]
[                                                                                                                                                     1/4*(pi*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L - pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L)*(C11_0 - C12_0)                                  -(pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L + (T - T_0)*alpha_11)*C11_0 - (pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L + (T - T_0)*alpha_11)*C12_0 - (pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L + (T - T_0)*alpha_33)*C13_0                                                                                                                                                                1/2*(pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L - pi*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L)*C44_0]
[                                                                                                                                                               1/2*(pi*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L - pi*cos(pi*x/L)*cos(pi*y/L)*e^(-t)*sin(pi*z/L)/L)*C44_0                                                                                                                                                                1/2*(pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L - pi*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L)*C44_0 -(pi*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L + (T - T_0)*alpha_33)*C33_0*(-0.500000000000000*sqrt(t) + 1) - (pi*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L + (T - T_0)*alpha_11)*C13_0 - (pi*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L + (T - T_0)*alpha_11)*C13_0]

In [16]:
div_stress = vector(SR,[0, 0, 0])

# Build div(σ)_i = ∂σ_ij / ∂x_j
for i in range(3):
    for j in range(3):
        div_stress[i] += diff(stress[j, i], [x,y,z][j])

show("Divergence of stress, must be used as body force with minus sign", div_stress)

'Divergence of stress, must be used as body force with minus sign' (-pi^2*C11_0*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L^2 - pi^2*C12_0*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L^2 - pi^2*C13_0*cos(pi*x/L)*e^(-t)*sin(pi*y/L)*sin(pi*z/L)/L^2 - 1/4*(pi^2*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L^2 + pi^2*cos(pi*x/L)*cos(pi*z/L)*e^(-t)*sin(pi*y/L)/L^2)*(C11_0 - C12_0) - 1/2*(pi^2*cos(pi*x/L)*cos(pi*y/L)*cos(pi*z/L)*e^(-t)/L^2 + pi^2*cos(pi*x/L)*e^(-t)*sin(pi*y/L)*sin(pi*z/L)/L^2)*C44_0, -pi^2*C11_0*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L^2 + pi^2*C12_0*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L^2 - pi^2*C13_0*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L^2 - 1/4*(pi^2*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L^2 - pi^2*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L^2)*(C11_0 - C12_0) - 1/2*(pi^2*cos(pi*y/L)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)/L^2 + pi^2*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L^2)*C44_0, -pi^2*C33_0*(-0.500000000000000*sqrt(t) + 1)*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L^2 + pi^2*C13_0*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L^2 + pi^2*C13_0*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L^2 - 1/2*(pi^2*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L^2 - pi^2*cos(pi*y/L)*e^(-t)*sin(pi*x/L)*sin(pi*z/L)/L^2)*C44_0 - 1/2*(pi^2*cos(pi*z/L)*e^(-t)*sin(pi*x/L)*sin(pi*y/L)/L^2 - pi^2*e^(-t)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L)/L^2)*C44_0)

In [17]:
b = -div_stress
b_simplified = [comp.simplify_full() for comp in b]
show("simplified body force in MMS: ", b_simplified)

'simplified body force in MMS: ' [1/4*(2*(2*pi^2*C13_0 + pi^2*C44_0)*cos(pi*x/L)*sin(pi*y/L)*sin(pi*z/L) + ((5*pi^2*C11_0 - pi^2*C12_0 + 2*pi^2*C44_0)*cos(pi*x/L)*cos(pi*y/L) + (pi^2*C11_0 + 3*pi^2*C12_0)*cos(pi*x/L)*sin(pi*y/L))*cos(pi*z/L))*e^(-t)/L^2,
 1/4*(2*(2*pi^2*C13_0 + pi^2*C44_0)*cos(pi*y/L)*sin(pi*x/L)*sin(pi*z/L) + ((5*pi^2*C11_0 - pi^2*C12_0 + 2*pi^2*C44_0)*cos(pi*y/L)*sin(pi*x/L) - (pi^2*C11_0 + 3*pi^2*C12_0)*sin(pi*x/L)*sin(pi*y/L))*cos(pi*z/L))*e^(-t)/L^2,
 -1/2*(1.0*pi^2*C33_0*sqrt(t)*cos(pi*z/L)*sin(pi*x/L)*sin(pi*y/L) - 2*(pi^2*C33_0 + pi^2*C44_0)*cos(pi*z/L)*sin(pi*x/L)*sin(pi*y/L) + ((2*pi^2*C13_0 + pi^2*C44_0)*cos(pi*y/L)*sin(pi*x/L) + (2*pi^2*C13_0 + pi^2*C44_0)*sin(pi*x/L)*sin(pi*y/L))*sin(pi*z/L))*e^(-t)/L^2]

In [18]:
def to_dealii_c(expr):
    """Convert a Sage expression to a C string using p[0],p[1],p[2]."""
    c_str = ccode(expr)
    # Replace coordinate symbols and time
    c_str = re.sub(r'\bx\b', 'p[0]', c_str)
    c_str = re.sub(r'\by\b', 'p[1]', c_str)
    c_str = re.sub(r'\bz\b', 'p[2]', c_str)
    # Sage may output 'pow' functions; deal.II's math needs std::pow or just multiplication
    # The expression already uses simple sin/cos, so it's fine.
    return c_str

In [19]:
function_signature = "Tensor<1,3> body_force(const Point<3> &p, const double t)"
function_body = function_signature + "\n{\n"
function_body += "  const double bx = " + to_dealii_c(b_simplified[0]) + ";\n"
function_body += "  const double by = " + to_dealii_c(b_simplified[1]) + ";\n"
function_body += "  const double bz = " + to_dealii_c(b_simplified[2]) + ";\n\n"
function_body += "  return Tensor<1,3>{{ bx, by, bz }};\n"
function_body += "}"

display(Markdown("### Generated deal.II function"))
print(function_body)

/home/wearetogether/miniforge3/envs/sage/lib/python3.11/site-packages/mpmath/libmp/libintmath.py:75: DeprecationWarning: bitcount function is deprecated
  warnings.warn("bitcount function is deprecated",


### Generated deal.II function

Tensor<1,3> body_force(const Point<3> &p, const double t)
{
  const double bx = (1.0/4.0)*(2*(2*pow(M_PI, 2)*C13_0 + pow(M_PI, 2)*C44_0)*sin(M_PI*p[1]/L)*sin(M_PI*p[2]/L)*cos(M_PI*p[0]/L) + ((pow(M_PI, 2)*C11_0 + 3*pow(M_PI, 2)*C12_0)*sin(M_PI*p[1]/L)*cos(M_PI*p[0]/L) + (5*pow(M_PI, 2)*C11_0 - pow(M_PI, 2)*C12_0 + 2*pow(M_PI, 2)*C44_0)*cos(M_PI*p[0]/L)*cos(M_PI*p[1]/L))*cos(M_PI*p[2]/L))*exp(-t)/pow(L, 2);
  const double by = (1.0/4.0)*(2*(2*pow(M_PI, 2)*C13_0 + pow(M_PI, 2)*C44_0)*sin(M_PI*p[0]/L)*sin(M_PI*p[2]/L)*cos(M_PI*p[1]/L) + (-(pow(M_PI, 2)*C11_0 + 3*pow(M_PI, 2)*C12_0)*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L) + (5*pow(M_PI, 2)*C11_0 - pow(M_PI, 2)*C12_0 + 2*pow(M_PI, 2)*C44_0)*sin(M_PI*p[0]/L)*cos(M_PI*p[1]/L))*cos(M_PI*p[2]/L))*exp(-t)/pow(L, 2);
  const double bz = -1.0/2.0*(1.0*pow(M_PI, 2)*C33_0*sqrt(t)*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L)*cos(M_PI*p[2]/L) - 2*(pow(M_PI, 2)*C33_0 + pow(M_PI, 2)*C44_0)*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L)*cos(M_PI*p[2]/L) + ((2*pow(M_PI, 2)*C13_0 + pow

In [20]:
actStress = stress[2,2] - sigma_th
macaulayBracketsActStress = (actStress + abs(actStress)) / 2 

f = -diff(omega,T) + A * (macaulayBracketsActStress / (sigma_th))^m * (1 / (1-omega))^m #Here Macaulay brackets are not implemented
f_simplified = f.simplify_full()
show("additional term for kinetic damage equation: ", f_simplified )

'additional term for kinetic damage equation: ' A*(-1/(0.5*sqrt(t) - 1))^m*(-1/2*(pi*C33_0*abs(L)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L) + L*sigma_th*abs(L)*e^t + 2*(C13_0*L*T*abs(L)*e^t - C13_0*L*T_0*abs(L)*e^t)*alpha_11 + (C33_0*L*T*abs(L)*e^t - C33_0*L*T_0*abs(L)*e^t)*alpha_33 - L*abs(-pi*C33_0*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L) - L*sigma_th*e^t - 2*(C13_0*L*T*e^t - C13_0*L*T_0*e^t)*alpha_11 - (C33_0*L*T*e^t - C33_0*L*T_0*e^t)*alpha_33 - (pi*C13_0*cos(pi*y/L)*sin(pi*x/L) + pi*C13_0*sin(pi*x/L)*sin(pi*y/L))*cos(pi*z/L) + (0.5*pi*C33_0*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L) + (0.5*C33_0*L*T*e^t - 0.5*C33_0*L*T_0*e^t)*alpha_33)*sqrt(t)) + (pi*C13_0*abs(L)*cos(pi*y/L)*sin(pi*x/L) + pi*C13_0*abs(L)*sin(pi*x/L)*sin(pi*y/L))*cos(pi*z/L) - (0.5*pi*C33_0*abs(L)*sin(pi*x/L)*sin(pi*y/L)*sin(pi*z/L) + (0.5*C33_0*L*T*abs(L)*e^t - 0.5*C33_0*L*T_0*abs(L)*e^t)*alpha_33)*sqrt(t))*e^(-t)/(L*sigma_th*abs(L)))^m

In [21]:
function_signature = "double kineticAddTerm(const Point<3> &p, const double t)"
function_body = function_signature + "\n{\n"
function_body += "  const double fadd = " + to_dealii_c(f_simplified) + ";\n"
function_body += "  return fadd;\n"
function_body += "}"

display(Markdown("### Generated deal.II function"))
print(function_body)

### Generated deal.II function

double kineticAddTerm(const Point<3> &p, const double t)
{
  const double fadd = A*pow(-1/(0.5*sqrt(t) - 1), m)*pow(-1.0/2.0*(M_PI*C33_0*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L)*sin(M_PI*p[2]/L)*fabs(L) + L*sigma_th*exp(t)*fabs(L) - L*fabs(M_PI*C33_0*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L)*sin(M_PI*p[2]/L) + L*sigma_th*exp(t) + 2*alpha_11*(C13_0*L*T*exp(t) - C13_0*L*T_0*exp(t)) + alpha_33*(C33_0*L*T*exp(t) - C33_0*L*T_0*exp(t)) - sqrt(t)*(0.5*M_PI*C33_0*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L)*sin(M_PI*p[2]/L) + alpha_33*(0.5*C33_0*L*T*exp(t) - 0.5*C33_0*L*T_0*exp(t))) + (M_PI*C13_0*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L) + M_PI*C13_0*sin(M_PI*p[0]/L)*cos(M_PI*p[1]/L))*cos(M_PI*p[2]/L)) + 2*alpha_11*(C13_0*L*T*exp(t)*fabs(L) - C13_0*L*T_0*exp(t)*fabs(L)) + alpha_33*(C33_0*L*T*exp(t)*fabs(L) - C33_0*L*T_0*exp(t)*fabs(L)) - sqrt(t)*(0.5*M_PI*C33_0*sin(M_PI*p[0]/L)*sin(M_PI*p[1]/L)*sin(M_PI*p[2]/L)*fabs(L) + alpha_33*(0.5*C33_0*L*T*exp(t)*fabs(L) - 0.5*C33_0*L*T_0*exp(t)*fabs(L))) + (M_PI*C13_0*sin(M_PI*p[0]/L)*si

In [22]:
#C11 = 1060e9 #for future use, now should bot be executed
#C33 = 36.5e9 
#C44 = 0.25e9 
#C12 = 180e9 
#C13 = 15e9 